# Pattern 6: Human-in-the-Loop (Approval Gate)

This notebook pauses the agent before the model step so a human can approve or edit input.
It uses static interruption with `interrupt_before=["model"]` and Claude Haiku 4.5.

```mermaid
flowchart LR
    P[Pending Action] --> G{Human Approval}
    G -->|Approve| X[Execute]
    G -->|Reject| S[Stop]
    G -->|Edit| E[Modify Input and Resume]
```

In [ ]:
from langchain.agents import create_agent
from langchain_anthropic import ChatAnthropic
from langgraph.checkpoint.memory import InMemorySaver

MODEL_NAME = "claude-haiku-4-5-20251001"
llm = ChatAnthropic(model=MODEL_NAME, temperature=0, max_tokens=1024)

agent = create_agent(
    model=llm,
    tools=[],
    checkpointer=InMemorySaver(),
    interrupt_before=["model"],
)

In [ ]:
config = {"configurable": {"thread_id": "approval-demo"}}

# First invoke pauses before model execution and returns interruption state.
paused = agent.invoke(
    {"messages": [{"role": "user", "content": "Draft a short email asking for a meeting tomorrow."}]},
    config=config,
)

print("Interrupted:", "__interrupt__" in paused)
print("Next step:", paused.get("next"))

In [ ]:
# Human review point: optionally replace None with a new input dict to modify the request.
approved_result = agent.invoke(None, config=config)
approved_result["messages"][-1].content

## How the code cells map to the pattern
Cell 2 sets up the interruptible Claude agent and the checkpoint store that lets the paused workflow resume safely.
Cell 3 triggers the interruption so you can see the approval boundary before the model generates a response.
Cell 4 resumes the workflow after human review, which is the key control point in this pattern.